# External comparison — BhashaBench-Legal, the FULL set

Every number in this project so far comes from a benchmark we built ourselves; the first
external run (Aug 2026) used a 1,500-question sample and its CI was wide. This runs all
24,365 questions, exact MCQ scoring, and keeps per-question rows so the result can be
sliced by language and subject afterwards.

**Settings:** GPU **T4 x2**, Internet **On**, secret `HF_TOKEN` (a read token on an account
that accepted the dataset terms on huggingface.co/datasets/bharatgenai/BhashaBench-Legal).
One model per session, ~5-6 h each: set `MODEL` and `LABEL` below.


In [ ]:
import os, subprocess, sys, json, re, time
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "transformers<5", "accelerate", "datasets"], check=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # T4 x2 -> DataParallel breaks placement
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

MODEL = "Qwen/Qwen2.5-3B-Instruct"          # second session: the shootout winner
LABEL = "base"

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # gated dataset

major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch})")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch build -- switch the accelerator to GPU T4 x2")
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("dtype:", DTYPE)


In [ ]:
from datasets import concatenate_datasets, get_dataset_config_names, load_dataset

DATASET = "bharatgenai/BhashaBench-Legal"
configs = get_dataset_config_names(DATASET, token=os.environ["HF_TOKEN"])
print("configs:", configs)
parts = []
for cfg in configs:
    ds = load_dataset(DATASET, cfg, token=os.environ["HF_TOKEN"])
    split = "test" if "test" in ds else list(ds.keys())[0]
    part = ds[split].add_column("lang", [cfg] * len(ds[split]))
    parts.append(part)
    print(f"{cfg}/{split}: {len(part)}")
bench = concatenate_datasets(parts)
SUBSET = 0                      # 0 = all questions
if SUBSET:
    bench = bench.shuffle(seed=0).select(range(SUBSET))
print(len(bench), "questions |", bench.column_names)

for col in ("question", "option_a", "option_b", "correct_answer"):
    assert col in bench.column_names, f"missing column: {col}"
golds = {str(r).strip().upper()[:1] for r in bench["correct_answer"]}
assert golds <= set("ABCD"), f"unexpected gold labels: {golds - set('ABCD')}"
SUBJECT_COL = next((c for c in bench.column_names
                    if c.lower() in ("subject", "topic", "domain", "subdomain", "category", "sub_domain")), None)
print("subject column:", SUBJECT_COL)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import gzip

LETTERS = ("A", "B", "C", "D")


def mcq_prompt(row):
    opts = chr(10).join(f"{L}. {row['option_' + L.lower()]}" for L in LETTERS
                        if row.get("option_" + L.lower()) not in (None, ""))
    return f"{row['question']}{chr(10)}{opts}{chr(10)}{chr(10)}Answer with the single letter of the correct option."


def run_mcq(model_id, label, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE, device_map={"": 0}).eval()
    correct = total = unparsed = 0
    rows, t0 = [], time.time()
    letter_re = re.compile(r"\b([ABCD])\b")
    for start in range(0, len(bench), batch_size):
        batch = bench.select(range(start, min(start + batch_size, len(bench))))
        texts = [tok.apply_chat_template([{"role": "user", "content": mcq_prompt(r)}],
                                         tokenize=False, add_generation_prompt=True) for r in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.pad_token_id)
        for row, text in zip(batch, tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            m = letter_re.search(text.upper())
            pred = m.group(1) if m else None
            unparsed += pred is None
            gold = str(row["correct_answer"]).strip().upper()[:1]
            correct += pred == gold
            total += 1
            rows.append({"pred": pred, "gold": gold, "lang": row["lang"],
                         "subject": row.get(SUBJECT_COL) if SUBJECT_COL else None})
        if (start // batch_size) % 50 == 0:
            done, el = start + len(batch), time.time() - t0
            print(f"[{label}] {done}/{len(bench)} acc {correct/max(1,total):.1%} elapsed {el:.0f}s eta {el/done*(len(bench)-done):.0f}s", flush=True)
    by_lang, by_subj = {}, {}
    for r in rows:
        by_lang.setdefault(r["lang"], []).append(r["pred"] == r["gold"])
        if r["subject"] is not None:
            by_subj.setdefault(str(r["subject"]), []).append(r["pred"] == r["gold"])
    summary = {"model": model_id, "accuracy": correct / total, "n": total, "unparsed": unparsed,
               "by_language": {k: sum(v) / len(v) for k, v in by_lang.items()},
               "by_subject": {k: {"n": len(v), "acc": sum(v) / len(v)} for k, v in sorted(by_subj.items())}}
    with gzip.open(f"/kaggle/working/bhashabench_full_{label}_rows.jsonl.gz", "wt", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + chr(10))
    json.dump(summary, open(f"/kaggle/working/bhashabench_full_{label}.json", "w"), indent=2)
    print(json.dumps({k: v for k, v in summary.items() if k != "by_subject"}, indent=2))
    del model
    torch.cuda.empty_cache()
    return summary


summary = run_mcq(MODEL, LABEL)


In [ ]:
# Standard error for one accuracy on n questions; the paired comparison against
# the other model is done on CPU afterwards from the saved rows.
import math
p, n = summary["accuracy"], summary["n"]
se = math.sqrt(p * (1 - p) / n)
print(f"{LABEL}: {p:.1%} on {n} questions, 95% CI +/-{1.96*se:.2%}; chance is 25%")
print("Download bhashabench_full_*.json and *_rows.jsonl.gz from the Output tab.")
